# Annual Data

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from datetime import datetime

In [206]:
df = pd.read_csv('../data/intermediate/annual.csv')
df = df.drop(columns=["Unnamed: 0"])

region_name_mapping = {
 'Архангельская область без АО':'Архангельская область',
 'Еврейская АО': 'Еврейская автономная область',
 'Кабардино-Балкария': 'Кабардино-Балкарская Республика',
 'Карачаево-Черкесия': 'Карачаево-Черкесская Республика',
 'Кемеровская область': 'Кемеровская область - Кузбасс',
 'Москва': 'г. Москва',
 'Санкт-Петербург': 'г. Санкт-Петербург',
 'Севастополь': 'г. Севастополь',
 'Северная Осетия': 'Республика Северная Осетия-Алания',
 'Тюменская область без АО': 'Тюменская область',
 'Чукотский АО': 'Чукотский автономный округ',
 'Якутия': 'Республика Саха (Якутия)'}

df['Region'] = df["Region"].replace(region_name_mapping)
ex_kia = df[['Region','year','total', 'drafted', 'contract', 'volunteers', 'pmc', 'prisoners', 'slavic', 'non_slavic']]
df = df[['Region', 'year','population', 'urban_share', 'median_income', 'share_poverty', '% Russians']]

ex = pd.read_csv('../data/intermediate/drafted.csv')
ex['mob_pc100'] = ex['mob_est']/ex['total']*100_000
ex.rename(columns={'region':'Region', 'total':'male_pop'}, inplace = True)
ex = ex[['Region', 'mob_pc100', 'male_pop', 'mob_est']] 
ex = ex.merge(df, on = 'Region', how = 'left')
ex = ex[ex['year'] == 2022].drop(columns = 'year')
ex = ex[ex['mob_pc100'] > 0]
ex = ex_kia.merge(ex, on = 'Region', how = 'left')

gov = pd.read_csv("../data/gov.csv")
today = pd.to_datetime(datetime.today().date())
gov['start'] = pd.to_datetime(gov['start'], errors='coerce')
gov['end'] = pd.to_datetime(gov['end'], errors='coerce')
gov['end'] = gov['end'].fillna(today)
gov = gov[(gov['end'] > '2022-12-01') & (gov['start'] < '2022-09-01')]
gov = gov[["Region", 'turnout21', 'ur21', 'region', 'nat_rep',
           'elections22', 'elections23', 'elections24', 'insider']].drop_duplicates()

region_name_mapping = {
 'Архангельская область без АО':'Архангельская область',
 'Еврейская АО': 'Еврейская автономная область',
 'Кабардино-Балкария': 'Кабардино-Балкарская Республика',
 'Карачаево-Черкесия': 'Карачаево-Черкесская Республика',
 'Кемеровская область': 'Кемеровская область - Кузбасс',
 'Москва': 'г. Москва',
 'Санкт-Петербург': 'г. Санкт-Петербург',
 'Севастополь': 'г. Севастополь',
 'Северная Осетия': 'Республика Северная Осетия-Алания',
 'Тюменская область без АО': 'Тюменская область',
 'Чукотский АО': 'Чукотский автономный округ',
 'Якутия': 'Республика Саха (Якутия)'}

gov['Region'] = gov["Region"].replace(region_name_mapping)
ex = ex.merge(gov, on = 'Region', how = 'left')

ex['elections'] = 0 
ex.loc[(ex['elections22'] == 1) & (ex['year'] == 2022), 'elections'] = 1
ex.loc[(ex['elections23'] == 1) & (ex['year'] == 2023), 'elections'] = 1
ex.loc[(ex['elections24'] == 1) & (ex['year'] == 2024), 'elections'] = 1

ex['non_draft'] = ex['volunteers'] + ex['pmc'] + ex['prisoners']

In [207]:
vars = ['drafted', 'non_draft']
for i in vars:
    ex[f'{i}_pc100'] = ex[i] / ex["male_pop"] * 100_000



pc100_cols = [f'{var}_pc100' for var in vars]
ex_long = ex.melt(
    id_vars=[col for col in ex.columns if col not in pc100_cols],  
    value_vars=pc100_cols, 
    var_name='category',    
    value_name='rate_pc100' 
)


ex_long = ex_long[ex_long['year'] != 2025]

In [220]:
model = smf.ols('rate_pc100 ~ mob_pc100 + urban_share + share_poverty +Q("% Russians") + ur21 +' \
' nat_rep + C(category) + C(year) + elections + insider',
                 data=ex_long).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             rate_pc100   R-squared:                       0.415
Model:                            OLS   Adj. R-squared:                  0.401
Method:                 Least Squares   F-statistic:                     29.04
Date:                 Ср, 27 авг 2025   Prob (F-statistic):           5.89e-46
Time:                        16:51:56   Log-Likelihood:                -2178.9
No. Observations:                 462   AIC:                             4382.
Df Residuals:                     450   BIC:                             4431.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

In [221]:
vars = ['slavic', 'non_slavic']
for i in vars:
    ex[f'{i}_pc100'] = ex[i] / ex["male_pop"] * 100_000

pc100_cols = [f'{var}_pc100' for var in vars]
ex_long = ex.melt(
    id_vars=[col for col in ex.columns if col not in pc100_cols],  
    value_vars=pc100_cols, 
    var_name='category',    
    value_name='rate_pc100' 
)


ex_long = ex_long[ex_long['year'] != 2025]

In [223]:
model = smf.ols('rate_pc100 ~ mob_pc100 + urban_share + share_poverty +Q("% Russians") + ur21 +' \
' nat_rep + C(category) + C(year) + elections + insider',
                data=ex_long).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             rate_pc100   R-squared:                       0.409
Model:                            OLS   Adj. R-squared:                  0.395
Method:                 Least Squares   F-statistic:                     28.31
Date:                 Ср, 27 авг 2025   Prob (F-statistic):           5.82e-45
Time:                        16:52:29   Log-Likelihood:                -2426.4
No. Observations:                 462   AIC:                             4877.
Df Residuals:                     450   BIC:                             4926.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [10]:
df['grants_per100k'] = df['other_transfers']/df['population']*100_000
df['log_grants'] = np.log(df['other_transfers'])
df['log_pop'] = np.log(df['population'])

vars = ['total', 'drafted', 'volunteers', 
        'pmc', 'prisoners', 'contract',
        'non_slavic', 'slavic',]

for var in vars:
    df[f'{var}_per100k'] = df[var]/df['population']*100_000



# Fiscal instruments 

## All soldiers

In [2]:
model = smf.ols('total_per100k ~ log_pop + grants_per100k + urban_share + median_income + share_poverty +Q("% Russians")',
                 data=df).fit()
print(model.summary())




NameError: name 'df' is not defined

## Drafted

In [13]:
model = smf.ols('volunteers_per100k ~ log_pop + grants_per100k + urban_share + median_income + share_poverty +Q("% Russians")',
                 data=df).fit()
print(model.summary())




                            OLS Regression Results                            
Dep. Variable:     volunteers_per100k   R-squared:                       0.069
Model:                            OLS   Adj. R-squared:                  0.046
Method:                 Least Squares   F-statistic:                     2.957
Date:                 Чт, 21 авг 2025   Prob (F-statistic):            0.00840
Time:                        17:33:17   Log-Likelihood:                -857.72
No. Observations:                 245   AIC:                             1729.
Df Residuals:                     238   BIC:                             1754.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          21.0522     12.871     

## Slavic

In [67]:
model = smf.ols('grants_per100k ~ log_pop + slavic_per100k + non_slavic_per100k +  urban_share +' \
' median_income + share_poverty + Q("% Russians")',
                 data=df).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:         grants_per100k   R-squared:                       0.408
Model:                            OLS   Adj. R-squared:                  0.390
Method:                 Least Squares   F-statistic:                     23.29
Date:                 Пн, 18 авг 2025   Prob (F-statistic):           5.93e-24
Time:                        01:39:20   Log-Likelihood:                -2173.5
No. Observations:                 245   AIC:                             4363.
Df Residuals:                     237   BIC:                             4391.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept           2648.3584   2812

# Predictors of draft

In [70]:
model = smf.ols('ex_nup_per100k ~ log_pop + urban_share + median_income + share_poverty + Q("% Russians") + insider',
                 data=df[df['year'] == 2022]).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:         ex_nup_per100k   R-squared:                       0.459
Model:                            OLS   Adj. R-squared:                  0.415
Method:                 Least Squares   F-statistic:                     10.59
Date:                 Пн, 18 авг 2025   Prob (F-statistic):           1.72e-08
Time:                        01:48:04   Log-Likelihood:                -534.21
No. Observations:                  82   AIC:                             1082.
Df Residuals:                      75   BIC:                             1099.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept        -618.2831    451.315     